Chart Analysis
Use Python to retrieve stock market data and analyze price changes.
1. Fetch both daily and hourly stock market data using a suitable API.
Visualize the data in several types of plots.
2. Plot long-term stock data over multiple years.
Create an interactive graph that allows the user to view different time ranges.
3. Start with Apple (AAPL) and plot its stock chart using daily data over several years.
4. Add moving averages to the stock charts, such as:
○ SMA20 and SMA50, or
○ SMA50 and SMA150
These should be overlaid on the main stock price chart.
5. Create a table for five stocks. Use default examples such as FAANG stocks plus
NVIDIA, but allow the user to change them.
For each stock, calculate the historical probability that the next day is positive if the
current day changes by:
+2%, +3%, +4%, +5%, and +6%.
Example: if Apple rises 5% today, what is the historical probability that the next day
closes positive?
Use the last 2 years of historical data.
6. Repeat the same analysis for negative daily moves, such as:
-2%, -3%, -4%, and -5%.
7. Repeat both of the above analyses again using 5 years of historical data.
8. Create a comparison table for the five selected stocks.
Include a simple correlation / collinearity analysis between them.
9. Plot a comparison chart for two to five stocks using daily data for one year.
Example: compare Apple and Amazon.
Normalize or rescale prices if needed so they can be compared clearly.
10. Create another table for the same five stocks showing the probability that the next day
is positive after:
● 2 consecutive positive days
● 3 consecutive positive days
● 4 consecutive positive days
● 5 consecutive positive days
● 6 consecutive positive days
11. Repeat the same analysis for consecutive negative days.
12. Screen all Nasdaq and NYSE stocks and identify the top 10 trending stocks, ranked
by daily percentage change.
13. Repeat the screening, but only include stocks with a market capitalization of $5 billion
or more.
This may require obtaining market cap data from an API or web scraping source.

In [1]:
import requests
import pandas as pd
import numpy as np

# Load API key from file
with open("api_keys/massive.txt", "r") as f:
    API_KEY = f.read().strip()

BASE_URL = "https://api.polygon.io"

In [5]:
def get_daily_data(ticker, start, end):
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}"
    params = {"adjusted": "true", "apiKey": API_KEY}
    r = requests.get(url, params=params).json()

    df = pd.DataFrame(r['results'])
    df['date'] = pd.to_datetime(df['t'], unit='ms')
    df.set_index('date', inplace=True)

    return df[['o','h','l','c','v']].rename(columns={
        'o':'open','h':'high','l':'low','c':'close','v':'volume'
    })

In [7]:
df_daily = get_daily_data("AAPL", "2020-01-01", "2025-01-01")
df

,open,high,low,close,volume
date,,,,,
2024-04-22 04:00:00,165.515,167.26,164.7700,165.84,48116443.0
2024-04-23 04:00:00,165.350,167.05,164.9200,166.90,49537761.0
2024-04-24 04:00:00,166.540,169.30,166.2100,169.02,48251835.0
2024-04-25 04:00:00,169.525,170.61,168.1511,169.89,50558329.0
2024-04-26 04:00:00,169.880,171.34,169.1800,169.30,44838354.0
...,...,...,...,...,...
2024-12-24 05:00:00,255.490,258.21,255.2900,258.20,23234705.0
2024-12-26 05:00:00,258.190,260.10,257.6300,259.02,27262983.0
2024-12-27 05:00:00,257.830,258.70,253.0600,255.59,42355321.0


# Massive limited free version

In [8]:
import requests
import pandas as pd

# Load API key from file
with open("api_keys/massive.txt", "r") as f:
    API_KEY = f.read().strip()

BASE_URL = "https://api.polygon.io"

def get_daily_data(ticker, start, end):
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }

    r = requests.get(url, params=params)
    data = r.json()

    print("status:", r.status_code)
    print("keys:", data.keys())
    print("resultsCount:", data.get("resultsCount"))
    print("queryCount:", data.get("queryCount"))
    print("ticker:", data.get("ticker"))

    if "results" not in data:
        print(data)
        raise ValueError("No 'results' returned from API")

    df = pd.DataFrame(data["results"])
    df["date"] = pd.to_datetime(df["t"], unit="ms")
    df = df.set_index("date").sort_index()

    df = df[["o", "h", "l", "c", "v"]].rename(columns={
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume"
    })

    return df

In [9]:
df = get_daily_data("AAPL", "2020-01-01", "2025-01-01")
print(df.head())
print(df.tail())
print(len(df))
print(df.index.min(), df.index.max())

status: 200
keys: dict_keys(['ticker', 'queryCount', 'resultsCount', 'adjusted', 'results', 'status', 'request_id', 'count'])
resultsCount: 176
queryCount: 176
ticker: AAPL
                        open    high       low   close      volume
date                                                              
2024-04-22 04:00:00  165.515  167.26  164.7700  165.84  48116443.0
2024-04-23 04:00:00  165.350  167.05  164.9200  166.90  49537761.0
2024-04-24 04:00:00  166.540  169.30  166.2100  169.02  48251835.0
2024-04-25 04:00:00  169.525  170.61  168.1511  169.89  50558329.0
2024-04-26 04:00:00  169.880  171.34  169.1800  169.30  44838354.0
                       open    high     low   close      volume
date                                                           
2024-12-24 05:00:00  255.49  258.21  255.29  258.20  23234705.0
2024-12-26 05:00:00  258.19  260.10  257.63  259.02  27262983.0
2024-12-27 05:00:00  257.83  258.70  253.06  255.59  42355321.0
2024-12-30 05:00:00  252.23  253.50  2

# Massive

In [10]:
def get_daily_data_chunked(ticker, year_starts):
    frames = []

    for start, end in year_starts:
        df_part = get_daily_data(ticker, start, end)
        frames.append(df_part)

    df = pd.concat(frames)
    df = df[~df.index.duplicated(keep="first")].sort_index()
    return df

In [11]:
ranges = [
    ("2020-01-01", "2020-12-31"),
    ("2021-01-01", "2021-12-31"),
    ("2022-01-01", "2022-12-31"),
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2025-01-01"),
]

df = get_daily_data_chunked("AAPL", ranges)
print(len(df))
print(df.index.min(), df.index.max())

status: 403
keys: dict_keys(['status', 'request_id', 'message'])
resultsCount: None
queryCount: None
ticker: None
{'status': 'NOT_AUTHORIZED', 'request_id': '69134c5b28b6d4d18258d9145635c8f0', 'message': "Your plan doesn't include this data timeframe. Please upgrade your plan at https://polygon.io/pricing"}


ValueError: No 'results' returned from API

FinnhubAPIException: FinnhubAPIException(status_code: 403): You don't have access to this resource.

In [13]:
print(finnhub_client.company_news('AAPL', _from="2020-06-01", to="2020-06-10"))

NameError: name 'finnhub_client' is not defined

# Old Massive

In [14]:
def get_hourly_data(ticker, start, end):
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/hour/{start}/{end}"
    params = {"adjusted": "true", "apiKey": API_KEY}
    r = requests.get(url, params=params).json()

    df = pd.DataFrame(r['results'])
    df['date'] = pd.to_datetime(df['t'], unit='ms')
    df.set_index('date', inplace=True)

    return df[['o','h','l','c','v']]

In [15]:
def add_moving_averages(df):
    df['SMA20'] = df['close'].rolling(20).mean()
    df['SMA50'] = df['close'].rolling(50).mean()
    return df

In [16]:
df['pct_change'] = df['close'].pct_change()

NameError: name 'df' is not defined